# MolQL selectors

MolViewSpec accepts eagerly built base MolQL expression trees for component and
color selectors and for primitive positions. PyMOL text is transpiled when the
state is created, so the MVSJ contains only JSON MolQL.

In [1]:
import { createBuilder, molql, molstarNotebook } from "../../molviewspec-ts/mod.ts";
import * as pymol from "../../molviewspec-ts/molviewspec/molql/transpilers/pymol/mod.ts";

## Build and transpile queries

Named MolQL arguments retain their canonical hyphenated spelling. The PyMOL
query is parsed immediately to the same base-language JSON representation.

In [3]:
const imatinib = molql.struct.generator.atomGroups({
  "chain-test": molql.core.rel.eq([
    molql.struct.atomProperty.macromolecular.label_asym_id(),
    "G",
  ]),
});

const imatinibN13 = molql.struct.generator.atomGroups({
  "chain-test": molql.core.rel.eq([
    molql.struct.atomProperty.macromolecular.label_asym_id(),
    "G",
  ]),
  "atom-test": molql.core.rel.eq([
    molql.struct.atomProperty.macromolecular.label_atom_id(),
    "N13",
  ]),
});

const bindingPocket = pymol.transpile("byres polymer within 5 of resn STI");
const thr315OG1 = pymol.transpile("chain A and resi 315 and name OG1");
const thr315 = pymol.transpile("chain A and resi 315");

## Use MolQL in an MVS state

In [9]:
const builder = createBuilder();
const structure = builder
  .download({ url: "https://files.wwpdb.org/download/1iep.cif" })
  .parse({ format: "mmcif" })
  .assemblyStructure();

const polymer = structure.component({ selector: "polymer" }).representation({
  type: "cartoon",
});
polymer.color({ color: "#8AA6C1" });
polymer.color({ color: "#B8497A", selector: molql.selector(bindingPocket) });
polymer.color({ color: "red", selector: molql.selector(thr315) });

structure
  .component({ selector: molql.selector(imatinib) })
  .representation({ type: "ball_and_stick" })
  .color({ color: "#F08A4B" });

structure
  .component({ selector: molql.selector(thr315) })
  .representation({ type: "ball_and_stick" })
  .color({ color: "red" });

structure.primitives().distance({
  start: molql.position(imatinibN13),
  end: molql.position(thr315OG1),
  color: "#F08A4B",
  dash_length: 0.2,
  label_template: "Imatinib N13–Thr315 OG1: {{distance}}",
});

const state = builder.getState({ title: "MolQL selectors" });
JSON.stringify(state)
// await molstarNotebook(state);

'{"kind":"single","root":{"kind":"root","children":[{"kind":"download","params":{"url":"https://files.wwpdb.org/download/1iep.cif"},"children":[{"kind":"parse","params":{"format":"mmcif"},"children":[{"kind":"structure","params":{"type":"assembly"},"children":[{"kind":"component","params":{"selector":"polymer"},"children":[{"kind":"representation","params":{"type":"cartoon"},"children":[{"kind":"color","params":{"color":"#8AA6C1"}},{"kind":"color","params":{"color":"#B8497A","selector":{"molql":{"head":{"name":"structure-query.generator.query-in-selection"},"args":{"0":{"head":{"name":"structure-query.modifier.expand-property"},"args":{"0":{"head":{"name":"structure-query.modifier.union"},"args":{"0":{"head":{"name":"structure-query.filter.within"},"args":{"0":{"head":{"name":"structure-query.generator.atom-groups"},"args":{"residue-test":{"head":{"name":"core.set.has"},"args":[{"head":{"name":"core.type.set"},"args":["A","C","T","G","U","DA","DC","DT","DG","DU","ALA","ARG","ASN","ASP"